# Module 4.4 — Pretrained Models: Standing on Giants' Shoulders

**Module 4: CNNs, Pretrained Models & Deep Dream | Week 2 | Lecture 4**

---

Training a CNN from scratch on CIFAR-10 got you ~70% accuracy after 10 epochs. State-of-the-art models trained on ImageNet (1.2 million images, 1000 classes) for days on powerful GPUs achieve 95%+.

Good news: you don't have to redo their work. PyTorch's `torchvision` ships with pretrained models you can load in one line. These models have already learned a rich set of features — edges, textures, shapes, objects — and you can use them directly.

## Glossary

Today's vocabulary is about borrowing a model someone else trained.

- **Pretrained** — a model whose weights were trained by someone else (usually on ImageNet). You can download the trained weights and use them directly.
- **ImageNet** (recap from 4.3) — the dataset most pretrained vision models were trained on. ~1.2 million images, 1000 classes.
- **Preprocessing** — the transformations applied to an image before feeding it to the model. Resize, crop, normalize with specific mean/std values that match what the pretrained model expects.
- **Softmax** (recap from M03) — the function that turns logits into probabilities that sum to 1. Used for multi-class classification outputs.
- **Forward hook** — a callback function PyTorch calls every time data passes through a chosen layer. We use hooks to peek at intermediate activations without modifying the model.
- **Feature extractor** — using a pretrained model as a fixed feature-computing pipeline (without retraining it). You feed in an image, grab the activations from some intermediate layer, and use those as features for your own downstream task.


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import urllib.request
import json

---

## Loading a Pretrained Model

One line gets you a ResNet-18 with weights trained on ImageNet.

In [ ]:
# Load pretrained ResNet-18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()  # Set to evaluation mode

params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18: {params:,} parameters")
print(f"Trained on: ImageNet (1.2M images, 1000 classes)")

This model has 11 million parameters, all pretrained. It already knows how to detect edges, textures, shapes, animal faces, cars, buildings — thousands of visual concepts.

---

## Classifying an Image

To use a pretrained model, you need to preprocess the image the same way it was trained. ImageNet models expect:
- 224×224 pixels
- Normalized with specific mean and std values

In [ ]:
# The standard ImageNet preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Download ImageNet class labels
url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
urllib.request.urlretrieve(url, 'imagenet_classes.txt')
with open('imagenet_classes.txt') as f:
    imagenet_classes = [line.strip() for line in f.readlines()]

print(f"Loaded {len(imagenet_classes)} ImageNet class names")
print(f"Examples: {imagenet_classes[:5]}")

In [ ]:
# Create a test image — use a CIFAR-10 image scaled up
from torchvision import datasets

cifar_raw = datasets.CIFAR10('data', train=False, download=True)
sample_img = cifar_raw[0][0]  # PIL image

# Preprocess
input_tensor = preprocess(sample_img).unsqueeze(0)

# Classify
with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)

# Top 5 predictions
top5_probs, top5_idx = probs.topk(5)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample_img)
plt.title(f'Input image (CIFAR-10: {["airplane","auto","bird","cat","deer","dog","frog","horse","ship","truck"][cifar_raw[0][1]]})',
          fontweight='bold')
plt.axis('off')

plt.subplot(1, 2, 2)
top5_names = [imagenet_classes[i] for i in top5_idx[0]]
top5_p = top5_probs[0].numpy()
bars = plt.barh(range(5), top5_p, color='#4A90D9', edgecolor='white')
plt.yticks(range(5), top5_names)
for bar, p in zip(bars, top5_p):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{p:.1%}', va='center', fontweight='bold')
plt.xlabel('Probability')
plt.title('Top-5 Predictions', fontweight='bold')
plt.xlim(0, max(top5_p) * 1.3)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

Note: CIFAR-10 images are tiny (32×32) so the model has less to work with. On full-resolution photos, pretrained models are remarkably accurate.

---

## What's Inside: The Feature Hierarchy

Last week we talked about feature hierarchies conceptually. Now let's actually see them. We'll extract feature maps from different depths of the network and visualize what they look like.

In [ ]:
# Extract features at different depths using hooks
activations = {}

def get_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks at different depths
model.conv1.register_forward_hook(get_hook('conv1'))      # very early
model.layer1.register_forward_hook(get_hook('layer1'))     # early
model.layer2.register_forward_hook(get_hook('layer2'))     # middle
model.layer3.register_forward_hook(get_hook('layer3'))     # late
model.layer4.register_forward_hook(get_hook('layer4'))     # very late

# Forward pass
with torch.no_grad():
    _ = model(input_tensor)

# Show dimensions at each layer
print(f"{'Layer':<10} {'Shape':>20} {'Channels':>10}")
print('=' * 45)
for name, act in activations.items():
    print(f"{name:<10} {str(list(act.shape)):>20} {act.shape[1]:>10}")

In [ ]:
# Visualize feature maps at each depth
fig, axes = plt.subplots(5, 8, figsize=(16, 10))

for row, (name, act) in enumerate(activations.items()):
    # Show first 8 channels
    for col in range(8):
        if col < act.shape[1]:
            fmap = act[0, col].numpy()
            axes[row, col].imshow(fmap, cmap='viridis')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{name}\n{act.shape[2]}×{act.shape[3]}',
                             fontsize=10, fontweight='bold')

plt.suptitle('Feature Maps at Different Depths (8 channels each)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

Notice the progression:
- **conv1** (early): large spatial resolution, features look like edge responses
- **layer1-2** (middle): smaller, more abstract patterns
- **layer3-4** (deep): tiny spatial dimensions, highly abstract — hard to interpret visually

This is the feature hierarchy in action. The early layers detect simple patterns, and each subsequent layer combines them into more complex features.

---

## Available Models in torchvision

PyTorch ships with many pretrained architectures. Here are the ones you're most likely to encounter:

In [ ]:
# Compare available models
model_info = [
    ('ResNet-18',  'resnet18',  11_689_512,  '69.8%'),
    ('ResNet-50',  'resnet50',  25_557_032,  '76.1%'),
    ('VGG-16',     'vgg16',    138_357_544,  '71.6%'),
    ('VGG-19',     'vgg19',    143_667_240,  '72.4%'),
    ('MobileNetV2','mobilenet_v2', 3_504_872, '71.9%'),
]

print(f"{'Model':<14} {'Parameters':>12} {'Top-1 Acc':>10} {'Notes'}")
print('=' * 60)
for name, _, params, acc in model_info:
    notes = ''
    if 'VGG' in name:
        notes = '← simple structure, good for Deep Dream'
    elif 'ResNet-50' in name:
        notes = '← strong general-purpose choice'
    elif 'Mobile' in name:
        notes = '← lightweight, fast'
    print(f"{name:<14} {params:>12,} {acc:>10} {notes}")

For the rest of this module:
- We'll use **VGG-16/19** for Deep Dream (simple sequential structure, easy to hook into)
- **ResNet** is the go-to for most practical tasks

---

## The Key Insight: Features Are Reusable

A model trained on ImageNet didn't just learn to classify 1000 specific categories. It learned a general visual vocabulary:

- **Layer 1:** Edges at various angles
- **Layer 2:** Corners, textures, color gradients
- **Layer 3:** Patterns, repeated textures, simple shapes
- **Layer 4:** Parts of objects (eyes, wheels, text)
- **Layer 5:** Objects and scenes

These features are useful for *any* visual task, not just the 1000 ImageNet classes. That's why pretrained models are so powerful — they give you a head start on any image problem.

Next session, we'll flip this idea on its head: instead of asking "what class is this image?", we'll ask **"what image would maximally activate this neuron?"** — and that's Deep Dream.

---

## Exercises

### Exercise 1: Try different models

Load VGG-16 (`models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)`) and classify the same image. Do ResNet-18 and VGG-16 agree on the top prediction?

### Exercise 2: Multiple images

Classify 10 different CIFAR-10 images with ResNet-18. For how many does the top-1 prediction match the CIFAR-10 label? (Note: ImageNet and CIFAR-10 have different class names, so match by concept, not exact string.)

### Exercise 3: Feature map sizes

Looking at the feature map dimensions from the hooks above: by what factor does the spatial size shrink from conv1 to layer4? How does the number of channels change? What's the tradeoff?

In [ ]:
# Exercises: Your code here


---

## Summary

- **Pretrained models** give you powerful feature extractors for free — load in one line
- **ImageNet preprocessing** is required: resize to 224, normalize with specific mean/std
- **Feature hierarchy** is real: early layers detect edges, deep layers detect objects
- **torchvision.models** has ResNet, VGG, MobileNet, and many more — all pretrained
- These features are **reusable** across tasks — that's what makes pretrained models valuable

---

**Next session:** Deep Dream — asking "what does a neuron want to see?"